In [1]:
import polars as pl
from pathlib import Path
import jellyfish
from nltk.corpus import names
import nltk
#nltk.download('names')
import re
import altair as alt

In [2]:
samples_df = pl.read_parquet("jails_pdfs_presentation.parquet")

In [3]:
cleaned = pl.read_parquet("jails_pdfs_cleanned.parquet")

In [7]:
yes = samples_df.filter(
    pl.col("Resulting Death?").str.contains("Yes")
)

In [8]:
test = cleaned.filter(
    pl.col("Cleaned Occurrences").str.contains("Rest")
)

In [9]:
test

Deceased Name,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Facility Name,Facility Type,RD Number,Phone Number,Address,Date,Time of Day,AM or PM,Occurrence,Table Contents,Injuries?,Resulting Death?,process,Report ID,OCR_Confidence,OCR_Word_Count,OCR_Text_Detected,Cook County?,Cleaned Facility Name,Cleaned Address,Zip Code,Cleaned Date,Cleaned Time of Day,Cleaned Occurrences
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],list[str],str,str,str,str,f64,i64,bool,str,str,str,str,str,str,str
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: St. Clair Count…","""Error creating ROI: zero-lengt…","""R.D. Number: ; """,""": 618-277-3505""","""\ddress: 700 North 5"" Street B…","""Date of Occurrence: 01/02/2021…","""me of Occurrence: 11:10 Oam. ""","""Error: less than two boxes fou…","[""Restraints Used —[ ""]","[""| 12/21/1995"", ""| 01/02/2021"", … ""Kiger, Bonham""]","""No injuries""","""No ""","""normal""","""Kiger, Bonham 1-2_p1""",87.941718,326,true,null,"""St Clair County Sheriffs Depar…","""700 North 5"" Street Belleville…","""62220""","""01/02/2021""","""11:10 Oam.""","""Restraints Used"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Knox County She…","""Error creating ROI: text ROI i…","""R.D, Number: KN21-0095 """,""": 309-345-3760""","""iddress: 152 S. Kellogg St. Ga…","""Shoo Date of Occurrence: 01/05…","""wy wee _ Time of Occurrence: 1…","""pM. ""","[""Restraints Used "", ""Sther (spocify): self harm ""]","[""aera seater"", ""Whiting, Preston Scott"", … ""cette""]","""No injuries""","""No ""","""normal""","""Whiting, Preston 1-5_p1""",82.486301,292,true,null,"""Knox County Sheriffs Departmen…","""152 S. Kellogg St. Galesburg I…","""61401""","""01/05/2021""","""1600""","""Restraints Used; Sther (spocif…"
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: Sangamon County…","""County ""","""oe R.D. Number: \gs we """,""": 217-753-6886""","""Address: #1 Sheriffs Plaza Spr…","""Date of Occurrence: 01/27/2021…","""_ Time of Occurrence: 1305 ""","""p.m, ""","[""Restraints Used ""]","[""“Michael Ledbetter"", ""1 08/06/1983"", … ""“Crim Damg_""]","""Yes, (briefly describe). Bumpe…","""Error: less than two boxes fou…","""normal""","""Ledbetter, Michael 1-27_p1""",85.127273,330,true,null,"""Sangamon County Jail""","""#1 Sheriffs Plaza Springfield …","""62701""","""01/27/2021""","""1305""","""Restraints Used"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: St. Clair Count…","""County ""","""R.D. Number: """,""": 618-277-3505""","""Address: 700 North 5"" Street B…","""Date of Occurrence: 01/13/2021…","""_ Time of Occurrence: 10:20 ""","""om. ""","[""Restraints Used ""]","[""(L2PANges |"", ""guzreoat"", … ""meee A me ai""]","""No injuries""","""No ""","""normal""","""Beard, Stephon 1-13_p1""",87.990033,301,true,null,"""St Clair County Sheriffs Depar…","""700 North 5"" Street Belleville…","""62220""","""01/13/2021""","""10:20""","""Restraints Used"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: Peoria County S…","""County ""","""R.D. Number: """,""": 309-697-7841""","""\ddress: 301 N Maxwell Rd Peor…","""Date of Occurrence: 01/08/2021…","""_ Time of Occurrence: 1700 ""","""p.m. ""","[""OC Spray Used "", ""Restraints Used ""]","[""Michael Knackmuhs"", ""05/01/1962"", … ""Hate Crim/Agg Battery""]","""No injuries""","""Error creating ROI: zero-lengt…","""normal""","""Knackmuhs, Michael 1-8_p1""",90.160131,306,true,null,"""Peoria County Sheriffs Dept""","""301 N Maxwell Rd Peoria IL 616…","""61604""","""01/08/2021""","""1700""","""OC Spray Used; Restraints Used"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""EE a) ~ & 1\\ een OO Facility …","

In [10]:
samples_df

Deceased Name,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Facility Name,Facility Type,RD Number,Phone Number,Address,Date,Time of Day,AM or PM,Occurrence,Table Contents,Injuries?,Resulting Death?,process,Report ID,OCR_Confidence,OCR_Word_Count,OCR_Text_Detected,Cook County?
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],list[str],str,str,str,str,f64,i64,bool,str
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Kane County She…","""County ""","""R.D. Number: ""","""__ Telephone #: 630-232-6840 ""","""\ddress: 37W755 IL Route 38 Su…","""Date of Occurrence: 01/17/2021…","""_ Time of Occurrence: 193\ ""","""No filled box found""","[""Other (specify): Resisting "", ""Restraints Used ""]","[""Ponce, David Julian"", ""04/13/1999"", … ""Crim Damage,DUI (X2),Resisting""]","""No injuries""","""No filled box found""","""normal""","""Ponce, David 1-17_p1""",89.921053,304,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Facility Name: Sangamon County…","""County ""","""R.D. Number: S _ = = ""","""\y a _. Telephone #: 717-753-6…","""Address: #1 Sheriffs Plaza Spr…","""Date of Occurrence: } — IG ~ 2…","""_ Time of Occurrence: O?7 40 ""","""aT. """,[],"[""Madk In, Ds iW..."", ""Pag i""]","""No injuries""","""Error: less than two boxes fou…","""normal""","""Macklin, Dion 1-16_p1""",84.018576,323,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: St. Clair Count…","""Error creating ROI: zero-lengt…","""R.D. Number: ; ""","""‘elephone #: 618-277-3505 ""","""\ddress: 700 North 5"" Street B…","""Date of Occurrence: 01/02/2021…","""me of Occurrence: 11:10 Oam. ""","""Error: less than two boxes fou…","[""Restraints Used —[ ""]","[""| 12/21/1995"", ""| 01/02/2021"", … ""Kiger, Bonham""]","""No injuries""","""No ""","""normal""","""Kiger, Bonham 1-2_p1""",87.941718,326,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: Tazewell County…","""County ""","""R.D.Number: ""","""_._ Telephone #: 309-478-5613 ""","""\ddress: 101 S. Capitol Pekin …","""Date of Occurrence: 1/10/2020 ""","""_ Time of Occurrence: 0435 ""","""a.m. ""","[""Restraints Used "", ""Other (specify): Control Used ""]","[""01/28/200 1"", ""- o1/ior2021"", … ""Domestic Battery""]","""No injuries""","""No filled box found""","""normal""","""Martin, Kaitlyn 1-10_p1""",88.321192,302,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Knox County She…","""Error creating ROI: text ROI i…","""R.D, Number: KN21-0095 ""","""____ Telephone #: 309-345-3760…","""iddress: 152 S. Kellogg St. Ga…","""Shoo Date of Occurrence: 01/05…","""wy wee _ Time of Occurrence: 1…","""pM. ""","[""Restraints Used "", ""Sther (spocify): self harm ""]","[""aera seater"", ""Whiting, Preston Scott"", … ""cette""]","""No injuries""","""No ""","""normal""","""Whiting, Preston 1-5_p1""",82.486301,292,true,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Cook County Dep…","""Caunty ""","""R.D. Number: DIV08-2023-17030 ""","""Telephone #: 773.674.3474 O IL…","""\ddress: 2700 S. California Di…","""Date of Occurrence: 7/31/2023 …","""of Occurrence: 9:16:00 PM ""","""Check time, no AM or PM for Co…","[""Fighting among ""]","[""Earnest R Rupert"", ""6/26/1980"", … ""7/28/2023""]","""No injuries""","""No filled box found""","""normal""","""FOIA - UO July 2023_p564""",88.549296,284,true,"""Cook County"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: Will County Adu…","""County ""","""R.D. Number; ""","""""","""address; QO ChicagoSt ———s—__…","""erect Date of Occurrence: MBW2…","""Why vial _ Time of Occurrence:…","""a.m. ""","[""Assault on Staff ""]","[""ae ee"", ""-Deiainees

In [11]:
processing = pl.read_parquet("processing_logs_full.parquet")

In [12]:
entries = ["Restraints Used\n", "Smattery"]


def clean_occurrences(entries):

    actual_occurrences = ["Suicide (method)", "Suicide (attempt)", "Homicide", "Homicide Attempt", "Escape", "Escape Attempt",
                "Fire", "Serious Injury", "Battery", "Riot of Rebellion", "Sex Offense", "Assault on Staff",
                "Assault among Detainees", "Fighting among Detainees", "Restraints Used", "OC Spray Used", "Other (specify)"]
    
    if entries.is_empty():
        return ["Error, no occurrence found"]

    cleaned_list = []
    for term in entries:
        best_choice = None
        highest_jaro = 0.0
        for compare_term in actual_occurrences:
            jaro_score = jellyfish.jaro_similarity(term, compare_term)
            if jaro_score > highest_jaro:
                highest_jaro = jaro_score
                best_choice = compare_term

        if best_choice in ["Suicide (method)", "Suicide (attempt)", "Other (specify)"]:
            # Keep the original term (possibly has colon text, like "Other (specify): Fight")
            cleaned_list.append(term)
        else:
            if best_choice not in cleaned_list:
                cleaned_list.append(best_choice)

    return cleaned_list

In [13]:
all_names = set(names.words())

test_list = ["Deaunte Campbell","3/12/2002","10/13/2021"]

def get_inmate_names(table_list):
    inmate_list = []
    pattern_punctuation = r"[^a-zA-Z0-9/ ]"
    for item in table_list:
        cleaned_item = re.sub(pattern_punctuation, '', item)
        tokens = cleaned_item.strip().split()
        for token in tokens:
            token = token.capitalize()
            if token in all_names:
                inmate_list.append(cleaned_item)
                break
    return inmate_list

get_inmate_names(test_list)

[]

In [14]:
occurrence_check = samples_df.select(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))
samples_df = samples_df.with_columns(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))
samples_df = samples_df.with_columns(pl.col("Table Contents").map_elements(get_inmate_names).alias("Inmate Names"))

C:\Users\ganon\AppData\Local\Temp\ipykernel_7316\1834402618.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  occurrence_check = samples_df.select(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))
C:\Users\ganon\AppData\Local\Temp\ipykernel_7316\1834402618.py:2: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samples_df = samples_df.with_columns(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))
C:\Users\ganon\AppData\Local\Temp\ipykernel_7316\1834402618.py:3: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samples_df = samples_df.with_columns(pl.col("T

In [15]:
# Clean facility name:
def clean_facility_name(df):
    """Clean facility names using Polars expressions"""
    return df.with_columns(
        pl.col('Facility Name')
        # First, clean OCR artifacts
        .str.replace_all(r'\\n|\\\\|\\/', ' ') 
        .str.replace_all(r'\n', ' ')
        .str.replace_all(r'[\\\/]+', ' ')
        .str.replace_all(r'\s+', ' ') 
        .str.strip_chars()
        # Extract facility name after any variation of "Name:"
        .str.extract(r'cility Name:\s*(.+)', 1)
        # Additional cleanup of extracted names
        .str.replace_all(r'[^a-zA-Z\s\'-]', ' ')  # Keep only letters, spaces, apostrophes, hyphens
        .str.replace_all(r'\s+', ' ')  # Clean multiple spaces again
        .str.strip_chars()  # Final trim
        # Stop at any word ending in "ddres" and remove that word + everything after
        .str.replace_all(r'\s+(ddress|ddrace).*$', '')
        .str.replace_all(r"(?i)\bSheriff's\b", "Sheriffs")
        .str.to_titlecase()
        .alias('Cleaned Facility Name')
    )

samples_df = clean_facility_name(samples_df)

# Date cleanning
def clean_date_occurrence(df):
    return df.with_columns(
        pl.col("Date")
          # OCR cleanup (keep forward slashes)
          .str.replace_all(r"\\n|\\\\", " ")
          .str.replace_all(r"\n", " ")
          .str.replace_all(r"\s+", " ")
          .str.strip_chars()
          # Extract the MM/DD/YYYY text
          .str.extract(r"Date of Occurrence:\s*([0-9]{1,2}/[0-9]{1,2}/(20[0-9]{2}))", 1)
          # Parse & re‐format to zero‐padded MM/DD/YYYY
          .str.strptime(pl.Date, "%m/%d/%Y", strict=False)
          .alias("temp_date")
    ).with_columns(
        # Only keep dates between 2000 and 2025, otherwise set to null
        pl.when(pl.col("temp_date").dt.year().is_between(2000, 2025))
        .then(pl.col("temp_date").dt.strftime("%m/%d/%Y"))
        .otherwise(None)
        .alias("Cleaned Date")
    ).drop("temp_date")

samples_df = clean_date_occurrence(samples_df)

In [16]:
samples_df = samples_df.drop(["OCR_Confidence", "OCR_Word_Count", "OCR_Text_Detected", "Occurrence", "Facility Name", "Date"])

samples_df = samples_df.with_columns(pl.col("Cleaned Occurrences").list.join(",").alias("Cleaned Occurrences"))

samples_df = samples_df.with_columns(pl.col("Inmate Names").list.join(",").alias("Inmate Names"))

samples_df = samples_df.with_columns(pl.col("Table Contents").list.join(",").alias("Table Contents"))

In [17]:
samples_df = samples_df.select(["Report ID", "Facility Type", "Cleaned Facility Name", "Address", "Phone Number", "Cleaned Date", "Time of Day", "AM or PM", "Table Contents", "Inmate Names", "Cleaned Occurrences", "Injuries?", "Resulting Death?", 'Deceased Cause', 'Deceased Date and Time', 'Deceased on Suicide Watch', 'Deceased Reporter', 'Deceased Examined by Physician', 'Deceased Signs of Illness'])

In [18]:
samples_df = samples_df.select(["Report ID", "Cleaned Occurrences", "Facility Type", "Cleaned Facility Name", "Address", "Phone Number", "Cleaned Date", "Time of Day", "AM or PM", "Table Contents", "Inmate Names", "Injuries?", "Resulting Death?", 'Deceased Cause', 'Deceased Date and Time', 'Deceased on Suicide Watch', 'Deceased Reporter', 'Deceased Examined by Physician', 'Deceased Signs of Illness'])

In [19]:
test = samples_df.filter(
    pl.col("Cleaned Occurrences").str.contains("Rest")
)

In [20]:
#test.write_excel("restraints_sample.xlsx")

In [21]:
test

Report ID,Cleaned Occurrences,Facility Type,Cleaned Facility Name,Address,Phone Number,Cleaned Date,Time of Day,AM or PM,Table Contents,Inmate Names,Injuries?,Resulting Death?,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Ponce, David 1-17_p1""","""Other (specify): Resisting ,Re…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""01/17/2021""","""_ Time of Occurrence: 193\ ""","""No filled box found""","""Ponce, David Julian,04/13/1999…","""Ponce David Julian""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Kiger, Bonham 1-2_p1""","""Restraints Used""","""Error creating ROI: zero-lengt…","""St Clair County Sheriffs Depar…","""\ddress: 700 North 5"" Street B…","""‘elephone #: 618-277-3505 ""","""01/02/2021""","""me of Occurrence: 11:10 Oam. ""","""Error: less than two boxes fou…","""| 12/21/1995,| 01/02/2021,Ag B…","""Ag Bate""","""No injuries""","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Martin, Kaitlyn 1-10_p1""","""Restraints Used,Other (specify…","""County ""","""Tazewell County Jail""","""\ddress: 101 S. Capitol Pekin …","""_._ Telephone #: 309-478-5613 ""","""01/10/2020""","""_ Time of Occurrence: 0435 ""","""a.m. ""","""01/28/200 1,- o1/ior2021,mee e…","""""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Whiting, Preston 1-5_p1""","""Restraints Used,Sther (spocify…","""Error creating ROI: text ROI i…","""Knox County Sheriffs Departmen…","""iddress: 152 S. Kellogg St. Ga…","""____ Telephone #: 309-345-3760…","""01/05/2021""","""wy wee _ Time of Occurrence: 1…","""pM. ""","""aera seater,Whiting, Preston S…","""Whiting Preston Scott""","""No injuries""","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Ledbetter, Michael 1-27_p1""","""Restraints Used""","""County ""","""Sangamon County Jail""","""Address: #1 Sheriffs Plaza Spr…","""__. Telephone #: 217-753-6886 ""","""01/27/2021""","""_ Time of Occurrence: 1305 ""","""p.m, ""","""“Michael Ledbetter,1 08/06/198…","""Michael Ledbetter""","""Yes, (briefly describe). Bumpe…","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FOIA - UO July 2023_p486""","""Other (specify): Rest chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""07/27/2023""","""_ Time of Occurrence: 9:4) ""","""p.m. ""","""Villa, Moises,12/26/2002,04/18…","""Villa Moises""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""FOIA - UO July 2023_p512""","""Restraints Used""","""County ""","""Peoria County Sheriffs Office""","""\ddress: 301 N. Maxwell Road P…","""__._- Telephone #: 309-697-784…","""07/29/2023""","""_ Time of Occurrence: 1730 ""","""p.m. ""","""Farmer, Sentaries,08/19/2004,0…","""""","""No injuries""","""Error creating ROI: zero-lengt…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""FOIA - UO July 2023_p521""","""Other (specify): ,Restraints U…","""County ""","""Peoria County Sheriffs Office""","""\ddress: 301 N. Maxwell Road P…","""__._- Telephone #: 309-697-784…","""07/29/2023""","""_ Time of Occurrence: 1130 ""","""a.m. ""","""Swartz, Henry,03/21/1996,02/27…","""Swartz Henry""","""No injuries""","""Error creating ROI: zero-lengt…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""


In [22]:
def clean_phone_number(df):
    pattern = r"#:"
    df = df.with_columns(
    pl.when(pl.col("Phone Number").str.contains(pattern))
    .then(pl.col("Phone Number").str.extract(r"#\s*(\S.*)"))
    .otherwise(pl.col("Phone Number"))
    .alias("Cleaned Phone Number")
    )

    return df

samples_df = clean_phone_number(samples_df)

In [23]:
def clean_time(df):
    pattern = r"Occurrence:"
    df = df.with_columns(
    pl.when(pl.col("Time of Day").str.contains(pattern))
    .then(pl.col("Time of Day").str.extract(r"e:\s*(\S.*)"))
    .otherwise(pl.col("Time of Day"))
    .alias("Cleaned Time of Day")
    )

    return df

samples_df = clean_time(samples_df)

In [24]:
def clean_phone_number(df):
    pattern = r"ss:"
    df = df.with_columns(
    pl.when(pl.col("Address").str.contains(pattern))
    .then(pl.col("Address").str.extract(r"s:\s*(\S.*)"))
    .otherwise(pl.col("Address"))
    .alias("Cleaned Address")
    )

    return df

In [25]:
def clean_address(df):
    pattern = r"ss:"
    df = df.with_columns(
    pl.when(pl.col("Address").str.contains(pattern))
    .then(pl.col("Address").str.extract(r"s:\s*(\S.*)"))
    .otherwise(pl.col("Address"))
    .alias("Cleaned Address")
    )

    return df

samples_df = clean_address(samples_df)

In [26]:
def get_zip_code(df):
    df = df.with_columns(
    pl.col("Cleaned Address").str.extract(r"(\d{5})").alias("Zip Code")
    )

    return df

samples_df = get_zip_code(samples_df)

In [27]:
def clean_injuries(df):
    pattern = r"ibe\)[;:]"
    df = df.with_columns(
    pl.when(pl.col("Injuries?").str.contains(pattern))
    .then(pl.col("Injuries?").str.extract(r"\)[;:]\s*(\S.*)"))
    .otherwise(pl.col("Injuries?"))
    .alias("Cleaned Injuries")
    )

    return df

samples_df = clean_injuries(samples_df)

In [28]:
df_sampled = samples_df.sample(n=100)

In [29]:
df_sampled

Report ID,Cleaned Occurrences,Facility Type,Cleaned Facility Name,Address,Phone Number,Cleaned Date,Time of Day,AM or PM,Table Contents,Inmate Names,Injuries?,Resulting Death?,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Cleaned Phone Number,Cleaned Time of Day,Cleaned Address,Zip Code,Cleaned Injuries
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""FOIA - UO Nov 2024_p60""","""Assault on Staff""","""County ""","""Christian County Jail""","""nddress: 301 W Franklin Taylor…","""Telephone #: 217-824-4961 """,null,"""wey lls bel heel —' ime of Occ…","""Error: less than two boxes fou…","""erence rere,Battery,05/21/1997""","""""","""Yes, (briefly describe); Set. …","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 217-824-4961""","""Aprox 8:22 Bam.""","""301 W Franklin Taylorville IL …","""62568""","""Set. Brownback, bloody nose, J…"
"""UO - FOIA October 2020_p42""","""Error, no occurrence found""","""County ""","""Coles County Safety Detention …","""Address: 701 7"" Street Charles…","""___ Telephone #: (217)348-7332…",null,"""_ Time of Occurrence: _( 23¢ x…","""am ""","""- Date ‘Gontincd:,~arreatg cha…","""DEA CO""","""Yes, (briefly describe): CyTT …","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": (217)348-7332""","""_( 23¢ xO""","""701 7"" Street Charleston IL) 6…","""61920""","""CyTT Cet Nir tt) C.F"""
"""FOIA - Feb 2024 UOs_p89""","""Dther (specify): Self-Harm ,Re…","""No filled box found""","""Jackson County Sheriffs Office""","""Address: 1001 Mulberry St Murp…","""___ Telephone #: 6186872292 ""","""02/06/2024""","""_ Time of Occurrence: 19° ""","""DIT ""","""Derrico Quinn,07/ 19 1989,02/0…","""Derrico Quinn""","""No injuries""","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 6186872292""","""19°""","""1001 Mulberry St Murphysboro I…","""62966""","""No injuries"""
"""FOIA - UO October 2018_p120""","""Battery""","""County ""","""Winnebago Criminal Justice Cen…","""Address: 650 W. State St. Rock…","""____ Telephone #: 815-319-6700…","""10/08/2018""","""_ Time of Occurrence: 1135 ""","""a.m. ""","""05/15/1993,""08/09/2018,Man/Del…","""Layne Shawnqual Johneil, Sand…","""Yes, (briefly describe): Munor…","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 815-319-6700""","""1135""","""650 W. State St. Rockford IL 6…","""61101""","""Munor scratch on inmate sander…"
"""FOIA - December 2024 UO Part 2…","""Battery""","""Caunty ""","""Cook County Department Of Corr…","""\ddress: 2700 S. California Av…","""Telephone #: 773.674.3474 O IL…","""12/08/2024""","""»f Occurrence: 9:21:00 AM ""","""Check time, no AM or PM for Co…","""Omari Whitaker,8/4/1998,10/24/…","""Omari Whitaker,KORY KOLBERT""","""No injuries""","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 773.674.3474""","""9:21:00 AM""","""2700 S. California Ave - CERMA…","""60608""","""No injuries"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FOIA - July 2024 UO Cook Co_p6…","""Battery""","""Caunty ""","""Cook County Department Of Corr…","""\ddress: 2700 S. California Di…","""Telephone #: 773.674.3474 @) I…","""07/10/2024""","""of Occurrence: 8:26:00 PM ""","""Check time, no AM or PM for Co…","""Kimonte T Cadge,12/6/1997,4/23…","""Carlos Cruz""","""No injuries""","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 773.674.3474""","""8:26:00 PM""","""2700 S. California Division 8 …","""60608""","""No injuries"""
"""UO - FOIA January 2023_p514""","""ther (specify): Hospital ""","""County ""","""Madison County Sheriffs Office""","""Address: 405 Randle Street Edw…","""__ Telephone #: 618-692-1065 ""","""01/19/2023""","""ine —— _ Time of Occurrence: 8…","""p.m. ""","""Terry, Abbe,09/07/1963,01/07/2…","""Terry Abbe""","""No injuries""","""N

In [30]:
count = samples_df.filter(
    pl.col("Cleaned Occurrences").str.contains("Error, no occurrence found")
)

In [31]:
count

Report ID,Cleaned Occurrences,Facility Type,Cleaned Facility Name,Address,Phone Number,Cleaned Date,Time of Day,AM or PM,Table Contents,Inmate Names,Injuries?,Resulting Death?,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Cleaned Phone Number,Cleaned Time of Day,Cleaned Address,Zip Code,Cleaned Injuries
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Macklin, Dion 1-16_p1""","""Error, no occurrence found""","""County ""","""Sangamon County Jail""","""Address: #1 Sheriffs Plaza Spr…","""\y a _. Telephone #: 717-753-6…",null,"""_ Time of Occurrence: O?7 40 ""","""aT. ""","""Madk In, Ds iW...,Pag i""","""""","""No injuries""","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 717-753-6886""","""O?7 40""","""#1 Sheriffs Plaza Springfield …","""62701""","""No injuries"""
"""Webster, Tiffany 1-9_p1""","""Error, no occurrence found""","""County ""","""Mchenry County Adult Correctio…","""Address: 2200 N. Seminary Ave,…","""___ Telephone #: 815-338-4741 ""","""01/09/2021""","""“ wily Leh ed Time of Occurren…","""p.m. ""","""pee oe,Webster, Tiffany,03/10/…","""Webster Tiffany""","""No injuries""","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 815-338-4741""","""2155""","""2200 N. Seminary Ave, i Woodst…","""60098""","""No injuries"""
"""Duff, Brian 1-21_p1""","""Error, no occurrence found""","""No filled box found""","""E Tov Ow Rh Yon""","""Facility Name: _ \LAW \TOF)\ \…","""__. Telephone #: | fi % - 8o4-…",null,""". City State -.... ff a ae} nd…","""No filled box found""","""Warted tobe rStreuinnc, tor ni…","""Warted tobe rStreuinnc tor nis…","""No injuries""","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": | fi % - 8o4- Lise""",""". City State -.... ff a ae} nd…","""Facility Name: _ \LAW \TOF)\ \…",null,"""No injuries"""
"""Crawford, Alexis 1-29_p1""","""Error, no occurrence found""","""County ""","""Sangamon County Adult Detentio…","""Address: #1 Sheriff's Plaza Sp…","""LY S SO ___ Telephone #: 753-6…",null,"""_ Time of Occurrence: x2 SU """,""".m ""","""gE ge Sg a en eee,Alexis Crawf…","""Alexis Crawford,Cass ss county…","""No injuries""","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 753-68 Ve""","""x2 SU""","""#1 Sheriff's Plaza Springfield…","""62701""","""No injuries"""
"""Loomis, Shane 1-5_p1""","""Error, no occurrence found""","""No filled box found""","""Wheside Courstu Shenfis Office…","""\ddress: Ur Ne) N C} | ru Ot :…","""_ Telephone #: 4 |! )- | IB—-U…",null,"""Time of Occurrence: / WF KE ""","""Error: less than two boxes fou…","""ioondeeda,anaes""","""""","""No injuries""","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 4 |! )- | IB—-Ued4""","""/ WF KE""","""Ur Ne) N C} | ru Ot : eee _Mom…",null,"""No injuries"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FOIA - UO July 2023_p384""","""Error, no occurrence found""","""Error: less than two boxes fou…","""Linton Op Io""","""Address: B10 Ee onl fun Ak. Co…","""Se _.. Telephone #: ( o\ 4 - B…",null,"""VS wwly WIGS _ Time of Occurre…","""Error: less than two boxes fou…","""somatee:,10 hy, Masterson :,ai…","""10 hy Masterson ,Tre Vahl ""","""No injuries""","""fo ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": ( o\ 4 - BUG. 7 Ly""","""\( Vie 8)""","""B10 Ee onl fun Ak. Coclube oe …",null,"""No injuries"""
"""FOIA - UO July 2023_p385""","""Error, no occurrence found""","""""","""Lax An Q Tey En Ee Ad Oe Fr""","""""","""_.. Telephone #: ( \S%- Qu -L """,null,"""wl vlale Co _ Time of Occurren…","""Error: less than two boxes fou…","""Pha a\nq. ~,TY 12\) \ a2,| ane…","""TY 12 a2""","""No injuries""","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": ( \S

In [32]:
samples_df = samples_df.with_columns(
    pl.when(pl.col("Cleaned Occurrences") == "")
    .then(pl.lit("Error, no occurrence found"))
    .otherwise(pl.col("Cleaned Occurrences"))
    .alias("Cleaned Occurrences")
)

def clean_other(df):
    pattern = r"specify"
    df = df.with_columns(
    pl.when(pl.col("Cleaned Occurrences").str.contains(pattern))
    .then(pl.col("Cleaned Occurrences").str.extract(r"\):\s*(\S.*)"))
    .otherwise(pl.col("Cleaned Occurrences"))
    .alias("Cleaned Occurrences")
    )

    return df

total_occurrence_result = (
    samples_df
    .with_columns(
        pl.col("Cleaned Occurrences").str.split(",").alias("Cleaned Occurrences")
    )
    .explode("Cleaned Occurrences")
    .filter(pl.col("Cleaned Occurrences").is_not_null())
    .select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(12)
)

total_occurrence_result = clean_other(total_occurrence_result)

total_occurrence_result = total_occurrence_result.filter(~pl.col("Cleaned Occurrences").is_in(["Error", " no occurrence found"]))

total_occurrence_chart = alt.Chart(total_occurrence_result).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
).properties(
    title="Most Common Occurrences Across All Jails"
)

total_occurrence_chart = alt.Chart(total_occurrence_result).mark_bar(
    color="#54728d"
).encode(
    x=alt.X('count:Q', title='Number of Cases'),
    y=alt.Y('Cleaned Occurrences:N', sort='-x', title='Occurrence'),
    tooltip=[
        alt.Tooltip('Cleaned Occurrences:N', title='Occurrence'),
        alt.Tooltip('count:Q', format=',d', title='Case Count')
    ]
).properties(
    width=600,
    height=400,
    title='Most Common Occurrences Across All Jails'
)

# Footnote as a separate text chart (keep your note)
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Occurrences may be concurrent. Missing 2,027 Occurrences (7.3%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,
    color="gray"
).encode(
    x=alt.value(-50),
    y=alt.value(440),
    text="text:N"
)

# Layer chart and footnote, remove box
total_occurrence_chart = alt.layer(
    total_occurrence_chart,
    footnote
).configure_view(
    stroke=None
)

total_occurrence_chart

alt.LayerChart(...)

In [33]:
pattern = r"\bspecify\b"

other_occurrence_table = samples_df.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)
    
other_occurrence_table = clean_other(other_occurrence_table)

other_occurrence_table = (
    other_occurrence_table.select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(10)
)

other_occurrence_chart = alt.Chart(other_occurrence_table).mark_bar(
    color="#da746d"
).encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence", sort='-x'),
    tooltip=[
        alt.Tooltip("Cleaned Occurrences:N", title="Occurrence"),
        alt.Tooltip("count:Q", format=",d", title="Case Count")
    ]
).properties(
    width=600,
    height=400,
    title="Most Common Other Occurrences Across All Jails"
)
other_occurrence_chart

alt.Chart(...)

In [34]:
samples_df = clean_other(samples_df)

In [35]:
top_prisons = (
    samples_df.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(11)
    )

top_prisons = top_prisons.filter(pl.col("Cleaned Facility Name") != "")

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 1,119 County Names (4.33%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(-50),
    y=alt.value(230),  # just below the chart (chart height + some offset)
    text="text:N"
)

top_prisons_chart = alt.Chart(top_prisons).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Occurrence Counts by Jails"
)

# Use layering to draw the footnote over/under the chart
top_prisons_chart = alt.layer(
    top_prisons_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

top_prisons_chart

alt.LayerChart(...)

In [48]:
top_counties = (
    samples_df
    .select(["Cleaned Facility Name", "Cleaned Occurrences"])
    .with_columns(
        pl.col("Cleaned Occurrences").str.split(",").alias("Cleaned Occurrences")
    )
    .explode("Cleaned Occurrences")
    .with_columns(
        pl.col("Cleaned Occurrences").str.strip_chars().alias("Cleaned Occurrences")
    )
    .filter(pl.col("Cleaned Occurrences").is_not_null())
)

def get_county_occurrences(current_jail):
    filtered_df = top_counties.filter(pl.col("Cleaned Facility Name") == current_jail)
    current_jail_result = (
    filtered_df.select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(12)
    )
    current_jail_result = current_jail_result.filter(~pl.col("Cleaned Occurrences").is_in(["Error", "no occurrence found"]))
    current_jail_chart = alt.Chart(current_jail_result).mark_bar(
    color="#54728d").encode(
    x=alt.X('count:Q', title='Number of Cases'),
    y=alt.Y('Cleaned Occurrences:N', sort='-x', title='Occurrence'),
    tooltip=[
        alt.Tooltip('Cleaned Occurrences:N', title='Occurrence'),
        alt.Tooltip('count:Q', format=',d', title='Case Count')
    ]
    ).properties(
    width=600,
    height=400,
    title=current_jail
    )
    return current_jail_chart

def get_county_occurrences_percent(current_jail):
    filtered_df = top_counties.filter(pl.col("Cleaned Facility Name") == current_jail)
    current_jail_result = (
        filtered_df.select(
            pl.col("Cleaned Occurrences").value_counts(sort=True)
        )
        .unnest("Cleaned Occurrences")
        .filter(~pl.col("Cleaned Occurrences").is_in(["Error", "no occurrence found"]))
    )
    total_count = current_jail_result["count"].sum()
    current_jail_result = current_jail_result.with_columns(
        (pl.col("count") / total_count * 100).alias("percentage")
    )
    current_jail_result = current_jail_result.sort("count", descending=True).head(10)
    current_jail_chart = alt.Chart(current_jail_result).mark_bar(
    color="#54728d").encode(
    x=alt.X('count:Q', title='Number of Cases'),
    y=alt.Y('Cleaned Occurrences:N', sort='-x', title='Occurrence'),
    tooltip=[
        alt.Tooltip('Cleaned Occurrences:N', title='Occurrence'),
        alt.Tooltip('count:Q', format=',d', title='Case Count')
    ]
    ).properties(
    width=600,
    height=400,
    title=current_jail
    )

    return current_jail_chart

cook_county = get_county_occurrences("Cook County Department Of Corrections")
cook_county_percent = get_county_occurrences_percent("Cook County Department Of Corrections")

kane_county = get_county_occurrences("Kane County Sheriffs Office")
kane_county_percent = get_county_occurrences_percent("Kane County Sheriffs Office")

madison_county = get_county_occurrences("Madison County Sheriffs Office")
madison_county_percent = get_county_occurrences_percent("Madison County Sheriffs Office")

lake_county = get_county_occurrences("Lake County Jail")
lake_county_percent = get_county_occurrences_percent("Lake County Jail")

peoria = get_county_occurrences("Peoria County Sheriffs Office")
peoria_percent = get_county_occurrences_percent("Peoria County Sheriffs Office")


In [49]:
cook_county_percent

alt.Chart(...)

In [50]:
kane_county_percent

alt.Chart(...)

In [51]:
madison_county_percent

alt.Chart(...)

In [52]:
lake_county_percent

alt.Chart(...)

In [53]:
peoria_percent

alt.Chart(...)

In [54]:
pattern = r"\bRestraints\b"

restraints_table = samples_df.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)

restraints_table = (
    restraints_table.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(21)
)

restraints_table = restraints_table.filter(pl.col("Cleaned Facility Name") != "")

restraints_chart = alt.Chart(restraints_table).mark_bar(
    color="#5a8664").encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x'),
    tooltip=[
        alt.Tooltip("Cleaned Facility Name:N", title="Jail"),
        alt.Tooltip("count:Q", format=",d", title="Case Count")
    ]
).properties(
    width=600,
    height=400,
    title="Top Jails for Restraint Usage"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 1,119 County Names (4.33%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(440),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
restraints_chart = alt.layer(
    restraints_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

restraints_chart

alt.LayerChart(...)

In [43]:
restraints_table = restraints_table.filter(pl.col("Cleaned Facility Name") != "null").join(top_counties, on="Cleaned Facility Name")

In [44]:
restraints_table = restraints_table.filter(
    pl.col("Cleaned Occurrences") == "Restraints Used"
)

restraints_table = restraints_table.unique()

In [45]:
restraints_perc_chart = alt.Chart(restraints_table.filter(pl.col("count_right") >= 100).with_columns(
    ((pl.col("count") / pl.col("count_right")).alias("perc")*100)
)).mark_bar().encode(
    x=alt.X("perc:Q", title="Percentage"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Restraint Usage as a Percent of Reports"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Only Jails with over 100 Reports Total"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(440),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
restraints_perc_chart = alt.layer(
    restraints_perc_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

ColumnNotFoundError: unable to find column "count_right"; valid columns: ["Cleaned Facility Name", "count", "Cleaned Occurrences"]

In [ ]:
restraints_perc_chart

alt.LayerChart(...)

In [55]:
pattern = r"\bFighting among Detainees\b"

fighting_table = top_counties.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)

fighting_table = (
    fighting_table.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(11)
)

fighting_table_table = fighting_table.filter(pl.col("Cleaned Facility Name") != "")

fighting_chart = alt.Chart(fighting_table).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Fighting among Detainees"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 214 Jail Names (12%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(250),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
fighting_chart = alt.layer(
    fighting_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [56]:
fighting_table

Cleaned Facility Name,count
str,u32
"""Cook County Department Of Corr…",4727
"""Madison County Sheriffs Office""",444
"""Kane County Sheriffs Office""",141
"""Lake County Jail""",137
"""Sangamon County Jail""",130
…,…
"""Winnebago County Jail""",72
"""Cook County Department Of Corr…",67
"""Cook County Department Of Corr…",47


In [ ]:
fighting_chart

alt.LayerChart(...)

In [ ]:
fighting_table = fighting_table.filter(pl.col("Cleaned Facility Name") != "null").join(top_counties, on="Cleaned Facility Name")

In [ ]:
fighting_table = fighting_table.filter(
    pl.col("Cleaned Occurrences") == "Restraints Used"
)

fighting_table = fighting_table.unique()

fighting_perc_chart = alt.Chart(fighting_table.filter(pl.col("count_right") >= 100).with_columns(
    ((pl.col("count") / pl.col("count_right")).alias("perc")*100)
)).mark_bar().encode(
    x=alt.X("perc:Q", title="Percentage"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Restraint Usage as a Percent of Reports"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Only Jails with over 100 Reports Total"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(230),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
fighting_perc_chart = alt.layer(
    fighting_perc_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [ ]:
fighting_perc_chart

alt.LayerChart(...)

In [ ]:
top_people = (
    samples_df.select(
        pl.col("Inmate Names").value_counts(sort=True)
    )
    .unnest("Inmate Names")
    .head(10)
    )

case_study = samples_df.filter(
    pl.col("Inmate Names") == "Ordonez Robert F"
)

In [ ]:
case_study

Report ID,Cleaned Occurrences,Facility Type,Cleaned Facility Name,Address,Phone Number,Cleaned Date,Time of Day,AM or PM,Table Contents,Inmate Names,Injuries?,Resulting Death?,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Cleaned Phone Number,Cleaned Time of Day,Cleaned Address,Zip Code,Cleaned Injuries
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""April 2021 FOIA Request_p67""","""Other (specify): Rest Chair ,O…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""04/30/2021""","""_ Time of Occurrence: 040! ""","""No filled box found""","""Ordonez, Robert F,08/12/1981,0…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""040!""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""April 2021 FOIA Request_p68""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""04/28/2021""","""_ Time of Occurrence: 034 ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""034""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""April 2021 FOIA Request_p69""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""04/27/2021""","""_ Time of Occurrence: 205: ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""205:""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""April 2021 FOIA Request_p70""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""04/26/2021""","""_ Time of Occurrence: 143: ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""143:""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""April 2021 FOIA Request_p71""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""04/25/2021""","""_ Time of Occurrence: 212: ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""212:""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Binder1_p279""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""06/22/2021""","""_ Time of Occurrence: 1210 hr ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez Robert F""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""": 630-232-6840""","""1210 hr""","""37W755 IL Route 38 Suite A Sai…","""60175""","""No injuries"""
"""Binder1_p280""","""Other (specify): Rest Chair ,R…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""06/23/2021""","""_ Time of Occurrence: 0342 hr ""","""No filled box found""","""Ordonez, Robert F.,08/12/1981,…","""Ordonez

In [ ]:
#samples_df.write_excel("7_10_sample.xlsx")

To-Do:
- Breakdown of largest counties